# Adaptacyjne planowanie

## Cel: Stworzysz w pełni autonomiczny system rezerwacji, który samodzielnie wykryje i naprawi błędy w nieprecyzyjnych danych wejściowych bez udziału człowieka.

### Czego się nauczysz:
1. Przeniesiesz sprawdzoną architekturę pętli decyzyjnej z branży gastronomicznej na wymagający grunt logistyki transportowej
2. Wyposażysz system w cyfrowe zmysły pozwalające na bezpieczną interakcję z bazami danych i historią klienta
3. Zaimplementujesz mechanizm wykrywania niejednoznaczności, który zmusi model do dopytania o brakujące szczegóły adresu
4. Oprogramujesz logikę wydobywania kluczowych informacji takich jak cena i czas z potoku tekstowego naturalnej rozmowy
5. Zbudujesz dynamicznie aktualizowany kontekst, który będzie pełnił rolę pamięci roboczej dla modelu w czasie rzeczywistym
6. Połączysz proces abstrakcyjnego wnioskowania z konkretnym działaniem poprzez rozbudowaną funkcję wykonawczą
7. Uruchomisz kompletną pętlę agenta, który samodzielnie przeprowadzi negocjacje od startu do sukcesu lub bezpiecznie przerwie misję

W poprzedniej lekcji zrobiliśmy kawał dobrej roboty. Zbudowaliśmy silnik `NextStep` dla restauracji, który poradził sobie z rezerwacją stolika. Mamy działający dowód na to, że nasza architektura po prostu działa.

Ale w inżynierii (i w życiu) prawdziwy sprawdzian przychodzi wtedy, gdy musisz przenieść rozwiązanie na zupełnie nowy grunt. I tutaj wracamy do naszej metodologii **bliźniaczych par**. Pamiętasz lekcję 1.4 🚕? Analizowaliśmy wtedy "na sucho" dialogi z dyspozytorem taxi. Rysowaliśmy diagramy, przewidywaliśmy problemy z brakiem taksówek czy fotelikami dla dzieci. Wtedy to była czysta teoria.

Teraz tchniemy w tamte schematy życie.

Tym razem jednak zmieniamy zasady gry. O ile przy restauracji prowadziłem Cię mocno za rękę, tak tutaj oddaję stery Tobie. Dlaczego? Bo głęboko wierzę, że prawdziwe kompetencje buduje się przez działanie, a nie przez oglądanie tutoriali. To jest ten moment, w którym musisz się trochę "pobrudzić" kodem, żeby naprawdę poczuć, jak ten system działa pod maską.

Podnosimy też poprzeczkę merytoryczną. Rezerwacja stolika była dość binarna (jest miejsce albo go nie ma). Zamawianie taksówki to domena, która zawiera więcej niejednoznaczności. Co zrobisz, gdy adres jest nieprecyzyjny? Jak zareaguje agent, gdy cena będzie za wysoka? To są ważne detale, którymi trzeba zarządzać.

Twój agent przestanie być tylko "głośnikiem" dla modelu językowego. Stanie się filtrem, który dba o jakość i poprawność danych, zanim w ogóle rozpocznie negocjacje.

Brzmi jak wyzwanie? I o to chodzi. Ale spokojnie – nie zostawiam Cię z tym. Przygotowałem masę podpowiedzi i gotowych fragmentów, które pomogą Ci, jeśli utkniesz.

Jestem pewien, że dasz sobie radę. Bierzmy się do pracy!

## Wczytujemy biblioteki

Jak zawsze, zaczynamy od fundamentów. Używamy `pydantic` do strukturyzacji myśli i `typing` do zachowania higieny kodu.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Union, List, Optional, Set
from enum import Enum
import json
import re
import helper as h
from helper_taxi import simulate_taxi_dispatcher # Symulator dyspozytora

## Definiujemy środowisko (*Mock Data*)

Zanim agent podniesie cyfrową słuchawkę, musi mieć dostęp do informacji. W systemie produkcyjnym podpięlibyśmy się tutaj pod Postgresa, Redisa czy API Google Maps. Na potrzeby edukacyjne wystarczą nam zwykłe słowniki Pythonowe, ale logika, która za nimi stoi, jest dokładnie taka sama jak w dużych systemach.

Przygotowaliśmy dla agenta trzy źródła danych, z których będzie musiał korzystać, aby nie popełnić błędu:

1.  **Baza klientów (`CUSTOMERS_DB`)** – to nasza **pamięć semantyczna**, analogicznie jak w przypadku rezerwacji restauracji. Mamy tu sztywne fakty o użytkowniku: preferencje płatności, status VIP czy wymogi specjalne (jak fotelik). Bez sprawdzenia tej bazy agent strzelałby w ciemno.
2.  **Historia przejazdów (`TAXI_HISTORY`)** – czyli **pamięć epizodyczna**. Zapisujemy tu konkretne zdarzenia z przeszłości. Skupiliśmy się na problemach (np. spóźnienie, brak terminala), bo w życiu tak to już działa, że negatywne doświadczenia pamiętamy znacznie mocniej i to one determinują nasze przyszłe decyzje. Jeśli klient raz spóźnił się przez korporację X na samolot, agent musi o tym wiedzieć, żeby nie popełnić tego błędu ponownie. Oczywiście w tej pamięci mogłyby być też pozytywne epizody, ale "czerwone flagi" są dla nas krytyczne.
3.  **Baza adresowa (`ADDRESS_DATABASE`)** – w świecie taksówek poprawny adres to być albo nie być całej usługi. Traktuj to jako uproszczony system walidacji (taki mock GIS/Map).

Szczególnie zwróć uwagę na ten ostatni element. Wprowadziliśmy tam celową pułapkę przy wpisie `"ul. Prosta"`. Adres jest oznaczony jako niejednoznaczny (`ambiguous`). Dlaczego? Bo w rzeczywistości, gdy pasażer rzuca hasło "Dworzec", a w mieście są dwa dworce, dobry dyspozytor dopyta "Który?". Nasz agent też musi to potrafić. Ta baza pozwala mu zweryfikować, czy wie wystarczająco dużo, by zlecić kurs, czy musi wrócić do użytkownika z pytaniem doprecyzującym.

In [ ]:
# `🧠 SEMANTIC MEMORY
class CustomerProfile(BaseModel):
    name: str
    phone: str
    preferred_payment: str
    max_price_pln: Optional[int] = None
    special_needs: List[str] = []
    vip: bool = False

# Baza klientów
CUSTOMERS_DB = {
    "CUST_001": CustomerProfile(
        name="Jan Kowalski",
        phone="+48123456789",
        preferred_payment="card",
        max_price_pln=150,
        special_needs=["child_seat_required"], # <-- WAŻNE
        vip=True
    ),
    "CUST_002": CustomerProfile(
        name="Anna Nowak",
        phone="+48987654321",
        preferred_payment="cash",
        max_price_pln=80,
        special_needs=["large_trunk_needed"]
    )
}

# `📚 EPISODIC MEMORY` - "Co się wydarzyło w przeszłości" (historia przejazdów)
TAXI_HISTORY = {
    ("CUST_001", "City Taxi"): {
        "date": "2024-01-15",
        "problem": "Kierowca spóźnił się 25 minut, klient stracił lot."
    },
    ("CUST_002", "Express Taxi"): {
        "date": "2024-02-10",
        "problem": "Kierowca nie miał terminala"
    }
}

# Baza adresowa - Zwróć uwagę na pułapkę przy "ul. Prosta"!
ADDRESS_DATABASE = {
    "ul. Złota 44": {"valid": True, "ambiguous": False},
    "ul. Marszałkowska 1": {"valid": True, "ambiguous": False},
    "ul. Prosta": {"valid": True, "ambiguous": True, 
                   "options": ["ul. Prosta 5", "ul. Prosta 15", "ul. Prosta 51"]},
    "ul. Prosta 5": {"valid": True, "ambiguous": False},   
    "ul. Prosta 15": {"valid": True, "ambiguous": False},  
    "ul. Prosta 51": {"valid": True, "ambiguous": False},  
    "Lotnisko Chopina": {"valid": True, "ambiguous": False},
}

## Struktury danych: cel, symulacja i stan

W tej komórce definiujemy trzy klasy, które posłużą nam za szkielet całej operacji. Warto się przy nich chwilę zatrzymać, bo wprowadzamy tu ważną nowość.

1.  **`TaxiScenario`** – to element czysto techniczny dla naszego symulatora. Ponieważ nie dzwonimy do prawdziwej korporacji, musimy mieć "pokrętło", którym ustawimy zachowanie drugiej strony. Czy dyspozytor ma od razu przyjąć zlecenie (`HAPPY_PATH`), czy może symulować brak kierowców? To pozwoli nam testować agenta w trudnych warunkach.
2.  **`TaxiBookingGoal`** – to po prostu **cel misji**. To jest nasza "kartka ze zleceniem", którą agent dostaje na starcie: skąd, dokąd, jaki budżet. Te dane są stałe – nie zmieniają się w trakcie rozmowy.

**I teraz najważniejsza rzecz:**

3.  **`ConversationState` (stan rozmowy)** – to jest ta nowość.

W poprzedniej lekcji (4.3) używaliśmy prostej listy `discoveries`, do której wrzucaliśmy luźne notatki ("Klient ma alergię"). To działało przy prostym procesie. Tutaj jednak proces jest zbyt złożony na luźne notatki.

Czym więc jest `ConversationState`? Możesz o tym myśleć jak o ustrukturyzowanej pamięci, która w czasie rzeczywistym buduje semantyczny obraz bieżącej sytuacji.

To nie jest baza danych na dysku. To jest notatnik, który agent trzyma w ręku podczas rozmowy i na bieżąco odhacza:
- "Czy sprawdziłem już historię klienta?" (bo nie chcę sprawdzać dwa razy)
- "Czy zwalidowałem adres?"
- "Czy cena została już ustalona?"

Która to jest warstwa pamięci? Jak myślisz? 

![](../../module2/images/memory_blue_print.jpg)

Skoro to pomaga procesować rzeczy krok do przodu, to brzmi jako *workflow memory*.

Dzięki temu, że trzymamy te informacje w strukturze Pydantic (a nie w luźnym tekście), będziemy mogli później bardzo precyzyjnie sterować zachowaniem agenta. Jeśli pole `addresses_validated` nie zawiera adresu docelowego, agent będzie wiedział – zero zgadywania – że jego priorytetem jest teraz walidacja adresu. To fundament pod dynamiczne budowanie promptu, które zrobimy za chwilę.

In [ ]:
# === SCENARIUSZE DO SYMULATORA ===
class TaxiScenario(str, Enum):
    HAPPY_PATH = "HAPPY_PATH"
    NO_CARS_AVAILABLE = "NO_CARS_AVAILABLE"
    LONG_WAIT_TIME = "LONG_WAIT_TIME" # Np. > 30 min
    
# === CEL MISJI (INPUT) ===
class TaxiBookingGoal(BaseModel):
    customer_id: str
    pickup_address: str
    destination: str
    time_preference: str
    max_wait_minutes: int
    max_price_pln: Optional[int]
    passenger_count: int = 1
    taxi_company: str
    scenario: TaxiScenario = TaxiScenario.HAPPY_PATH 

# 🎯 WORKFLOW MEMORY
# STAN ROZMOWY (MUTABLE STATE) 
class ConversationState(BaseModel):
    # Tracking co już zrobiliśmy
    customer_fetched: bool = False
    customer_name: Optional[str] = None
    history_checked: bool = False
    history_problem: Optional[str] = None
    addresses_validated: Set[str] = set()
    special_needs: List[str] = []
    is_vip: bool = False
    max_price: Optional[int] = None
    
    # Rozmowa
    conversation_started: bool = False
    eta_minutes: Optional[int] = None
    price_quoted: Optional[int] = None
    special_confirmed: bool = False
    dispatcher_asked_name: bool = False
    name_given: bool = False
    driver_assigned: bool = False
    booking_id: Optional[str] = None
    
    # Do symulatora
    driver_status_db: dict = {}
    scenario: TaxiScenario = TaxiScenario.HAPPY_PATH

## 🧠 Zadanie 4.4.1: Zdefiniuj "zmysły" agenta

W poprzednim kroku zdefiniowaliśmy "świat" (nasze bazy danych). Ale agent, tak jak człowiek z zawiązanymi oczami, nie widzi tych danych bezpośrednio. Potrzebuje narzędzi – swoistych zmysłów – aby po nie sięgnąć.

Twoim zadaniem jest zaprojektowanie interfejsu (warstwy narzędzi), który połączy "mózg" agenta z naszymi bazami danych.

Musisz przeanalizować strukturę danych `CUSTOMERS_DB`, `TAXI_HISTORY` oraz `ADDRESS_DATABASE` i stworzyć dla agenta zestaw klas Pydantic, które pozwolą mu wydobyć z nich informacje.

### Twoim zadaniem jest:
Zdefiniowanie zestawu klas dziedziczących po `BaseModel`, które pokryją następujące zapotrzebowanie operacyjne:

1.  **Identyfikacja klienta:** Agent musi mieć sposób, by pobrać profil z `CUSTOMERS_DB` na podstawie ID.
2.  **Bezpieczeństwo (Historia):** Potrzebujemy narzędzia do sprawdzenia w `TAXI_HISTORY`, czy dana para (klient + korporacja) nie miała w przeszłości problemów.
3.  **Weryfikacja adresu:** To nowość i kluczowy element tej lekcji. Agent nie może ufać "na słowo", że ulica istnieje. Stwórz narzędzie, które pozwoli mu odpytać `ADDRESS_DATABASE`.
4.  **Status zamówienia:** Narzędzie do sprawdzenia, co dzieje się z aktywnym zleceniem (np. czy kierowca już jedzie).
5.  **Interakcja:** Standardowe narzędzia do prowadzenia rozmowy (`Say`), zakończenia sukcesem (`Complete`) lub porażką (`Abandon`).

**Wymaganie architektoniczne:**
Aby nasz Router (który napiszemy później) działał poprawnie, każda z Twoich klas **musi** posiadać pole `action` typu `Literal`. To będzie nasz techniczny wyróżnik, dzięki któremu system będzie wiedział, czy ma do czynienia z pytaniem o adres, czy z kończeniem rozmowy.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Jak o tym myśleć?

Nie zgaduj nazw pól. Spójrz na definicje naszych baz danych (słowników Pythonowych) wyżej.
*   Jeśli kluczem w `ADDRESS_DATABASE` jest string z adresem, to Twoje narzędzie do walidacji musi przyjmować właśnie taki parametr (np. `address: str`).
*   Jeśli kluczem w `TAXI_HISTORY` jest para `(customer_id, taxi_company)`, to Twoje narzędzie musi prosić model o wygenerowanie obu tych informacji.

Pamiętaj też o `Literal`. To "etykieta" dla kodu. Wzorzec wygląda tak:

```python
class NameTool(BaseModel):
    # To pole służy tylko do routingu - model ma tu tylko jedną opcję wyboru
    action: Literal["unique_name"] = "unikalna_nazwa_akcji"
    
    # To są faktyczne dane, o które prosi model
    param_1: str = Field(..., description="Opis dla modelu po co to jest")
```

Teraz przełóż to na 7 konkretnych klas.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
# === AKCJE (TOOLS) ===

class QueryCustomerData(BaseModel):
    action: Literal["query_customer"] = "query_customer"
    customer_id: str = Field(..., description="ID klienta do pobrania danych")

class CheckTaxiHistory(BaseModel):
    action: Literal["check_taxi_history"] = "check_taxi_history"
    customer_id: str = Field(..., description="ID klienta")
    taxi_company: str = Field(..., description="Nazwa korporacji taksówkarskiej")

class ValidateAddress(BaseModel):
    action: Literal["validate_address"] = "validate_address"
    address: str = Field(..., description="Adres do walidacji w bazie danych")

class CheckDriverStatus(BaseModel):
    action: Literal["check_driver_status"] = "check_driver_status"
    booking_id: str = Field(..., description="ID zamówienia")

class Say(BaseModel):
    action: Literal["say"] = "say"
    utterance: str = Field(..., description="Co agent ma powiedzieć do dyspozytora")

class Complete(BaseModel):
    action: Literal["complete"] = "complete"
    summary: str = Field(..., description="Podsumowanie rezerwacji")

class Abandon(BaseModel):
    action: Literal["abandon"] = "abandon"
    reason: str = Field(..., description="Powód niepowodzenia")
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.4.2: Integracja silnika decyzyjnego (`NextStep`)

Mamy zdefiniowane środowisko (dane) oraz "ręce" agenta (narzędzia). Teraz musimy włożyć mu do głowy "centrum decyzyjne".

Nie pozwalamy modelowi działać chaotycznie. Wymuszamy na nim strukturę myślenia: najpierw analiza, potem decyzja. To nadal nie gwarantuje, że zawsze będzie działał poprawnie, ale pomaga mu uruchomić tryb pracy, który nie tylko podnosi jakość wyników, lecz także umożliwia nam obserwację jego rozumowania. Dzięki temu zarówno model, jak i jego sposób podejmowania decyzji stają się dla nas bardziej transparentne.

Twoim zadaniem jest zdefiniowanie klasy `NextStep`, która będzie sercem naszej pętli. To ona połączy proces wnioskowania (Reasoning) z wyborem konkretnego narzędzia (*Routing*).


```

┌─────────────────────────────────────────────────────────────────────┐
│                           START KONWERSACJI                         │
└─────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
┌─────────────────────────────────────────────────────────────────────┐
│  ┌───────────────────────────────────────────────────────────────┐  │
│  │              SITUATION_ANALYSIS (Analiza sytuacji)            │  │
│  │         "Co wiem? Czego brakuje? Jaki następny krok?"         │  │
│  └───────────────────────────────────────────────────────────────┘  │
│                                   │                                 │
│                                   ▼                                 │
│  ┌───────────────────────────────────────────────────────────────┐  │
│  │                    🤔 WYBÓR NEXT_ACTION                       │  │
│  └───────────────────────────────────────────────────────────────┘  │
│         │          │          │          │          │               │
│         ▼          ▼          ▼          ▼          ▼               │
│  ┌──────────┬──────────┬──────────┬──────────┬──────────┐           │
│  │   Query  │  Check   │  Valid-  │  Check   │   Say    │           │
│  │ Customer │  Taxi    │   ate    │  Driver  │          │           │
│  │   Data   │ History  │ Address  │  Status  │          │           │
│  └────┬─────┴────┬─────┴────┬─────┴────┬─────┴────┬─────┘           │
│       │          │          │          │          │                 │
│       ▼          ▼          ▼          ▼          ▼                 │
│  ┌─────────────────────────────────┐  ┌─────────────────┐           │
│  │        🗄️ BAZA DANYCH           │  │  📞 DYSPOZYTOR  │           │
│  │    (wynik operacji/dane)        │  │   (odpowiedź)   │           │
│  └────────────────┬────────────────┘  └────────┬────────┘           │
│                   │                            │                    │
│                   └──────────┬─────────────────┘                    │
│                              │                                      │
│  🔁 PĘTLA                    │ (powrót do analizy)                  │
└──────────────────────────────┼──────────────────────────────────────┘
                               │
          ┌────────────────────┼────────────────────┐
          │                    │                    │
          ▼                    │                    ▼
┌──────────────────┐           │         ┌──────────────────┐
│   ✅ COMPLETE    │           │         │    ❌ ABANDON    │
│                  │           │         │                  │
│  summary: str    │           │         │  reason: str     │
│  (podsumowanie   │           │         │  (powód          │
│   rezerwacji)    │           │         │   niepowodzenia) │
└──────────────────┘           │         └──────────────────┘
                               │
                    (kontynuuj pętlę)
```



### Twoim zadaniem jest:
Zdefiniowanie klasy `NextStep`, która spełnia dwa kluczowe założenia architektoniczne:
1.  **Wymusza "Chain of Thought":** Musi posiadać pole `situation_analysis`, w którym model krótko przeanalizuje obecny stan (np. "Mam dane klienta, ale nie mam adresu, muszę go zwalidować").
2.  **Polimorficzny wybór akcji:** Musi posiadać pole `next_action`, które pozwoli modelowi wybrać **dokładnie jedno** z narzędzi zdefiniowanych w poprzednim zadaniu.

**Wskazówka techniczna:**
Aby Pydantic poprawnie rozpoznał, którą klasę narzędzia wybrał model, musisz użyć `Union` zawierającego wszystkie Twoje klasy akcji, oraz wskazać pole `action` jako dyskryminator (`discriminator="action"`).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

To jest moment, w którym łączymy wszystkie "klocki" z poprzedniego zadania w jeden worek decyzyjny.

Twój kod powinien wyglądać schematycznie tak:

```python
class NextStep(BaseModel):
    situation_analysis: str = Field(...)
    next_action: Union[ToolA, ToolB, ToolC...] = Field(..., discriminator="action")
```

Pamiętaj, aby w `Union` wymienić **wszystkie 7 klas**, które zostały zrobione przed chwilą w zadaniu 4.4.1 (`QueryCustomerData`, `CheckTaxiHistory`, `ValidateAddress`, `CheckDriverStatus`, `Say`, `Complete`, `Abandon`). Bez tego agent nie będzie miał dostępu do niektórych swoich umiejętności.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
# === DECISION ENGINE ===

class NextStep(BaseModel):
    situation_analysis: str = Field(
        ...,
        description="Brief analysis of the current situation. What do I know? What is missing?"
    )
    next_action: Union[
        QueryCustomerData,
        CheckTaxiHistory,
        ValidateAddress,
        CheckDriverStatus,
        Say,
        Complete,
        Abandon
    ] = Field(..., discriminator="action")
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## Głowne elementy 

- ✅ `NextStep` już mamy
- teraz czas na `dispatch()`


```text
    ┌───────────────────────────────────────────────────────────────┐
    │                                                               │
    │   ┌─────────┐    ┌───────────┐    ┌───────────┐    ┌──────┐   │
    │   │   LLM   │───▶│ NextStep  │───▶│ dispatch()│───▶│result│   │
    │   └─────────┘    └───────────┘    └───────────┘    └──────┘   │
    │        ▲                                              │       │
    │        │                                              │       │
    │        └──────────────────────────────────────────────┘       │
    │                    (jeśli NIE Terminal)                       │
    │                                                               │
    └───────────────────────────────────────────────────────────────┘
    
    STOP gdy: Complete ✅ lub Abandon ❌
    
``` 

## 🧠 Zadanie 4.4.3: Dispatcher - warstwa danych

Mamy mózg (`NextStep`), mamy narzędzia. Teraz musimy połączyć kable. Twoim zadaniem jest napisanie pierwszej części funkcji `dispatch`.

To miejsce, w którym "intencja" zamienia się w "działanie".
Skupimy się na narzędziach **informacyjnych** (Read-Only): `QueryCustomerData` oraz `CheckTaxiHistory`.

Nie chodzi tylko o pobranie danych i wyświetlenie ich. Kluczową rolą tej funkcji jest **aktualizacja stanu (`ConversationState`)**. Agent jest bezstanowy – to Ty, jako inżynier, musisz zadbać o to, by po pobraniu danych "odhaczyć" ten fakt w pamięci roboczej. Bez tego agent utknie w pętli, w kółko pytając o to samo.

### Twoim zadaniem jest:
Zaimplementowanie obsługi dwóch pierwszych akcji wewnątrz funkcji `dispatch`.

1.  **Obsługa `QueryCustomerData`:**
    *   **Zabezpieczenie (Idempotency):** Sprawdź w `state`, czy dane nie zostały już pobrane. Jeśli tak, zwróć ostrzeżenie, aby agent przeszedł dalej.
    *   **Pobranie:** Wyciągnij obiekt z `CUSTOMERS_DB`.
    *   **Mutacja Stanu:** Zaktualizuj pola w `state` (`customer_name`, `is_vip`, `special_needs`, etc.) i ustaw flagę `customer_fetched = True`.
    *   **Zwrot:** Zwróć czytelny string z danymi (np. "✅ CUSTOMER DATA: ...").

2.  **Obsługa `CheckTaxiHistory`:**
    *   **Zabezpieczenie:** Podobnie jak wyżej, sprawdź, czy historia była już sprawdzana.
    *   **Klucz:** Zauważ, że kluczem w `TAXI_HISTORY` jest krotka (tuple): `(customer_id, taxi_company)`. Musisz ją zbudować.
    *   **Mutacja Stanu:** Jeśli znajdziesz problem, zapisz go w `state.history_problem`. Ustaw flagę `history_checked = True`.
    *   **Zwrot:** Zwróć informację o problemie lub o czystej historii.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Dlaczego "Guard Clause" jest ważny?
Modele LLM czasem się zapętlają. Mogą uznać, że "dla pewności" sprawdzą klienta jeszcze raz. Twoim zadaniem jest wykrycie tego w kodzie.

Struktura Twojego kodu powinna wyglądać tak:

```python
if isinstance(action, QueryCustomerData):
    # 1. Guard Clause (Blokada pętli)
    if state.customer_fetched:
        return f"⚠️ ERROR: Customer data already fetched for {state.customer_name}. Move to the next step!"
    
    # 2. Logic & Side Effects
    profile = CUSTOMERS_DB.get(action.customer_id)
    # ... update state fields ...
    state.customer_fetched = True
    
    # 3. Return Feedback
    return f"✅ CUSTOMER DATA: {profile.name}..."
```

Pamiętaj też, że klucze słownika w Pythonie mogą być tuplami. Aby odpytać historię, zrób: `key = (action.customer_id, action.taxi_company)`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
def dispatch(action, goal: TaxiBookingGoal, state: ConversationState) -> str:
    
    # === QueryCustomerData ===
    if isinstance(action, QueryCustomerData):
        # Loop protection
        if state.customer_fetched:
            return f"⚠️ ERROR: Customer data already fetched! Name: {state.customer_name}. Move to the next step!"
        
        profile = CUSTOMERS_DB.get(action.customer_id)
        if not profile:
            return f"❌ Error: Customer {action.customer_id} not found"
        
        # State Mutation (Side Effects)
        state.customer_fetched = True
        state.customer_name = profile.name
        state.is_vip = profile.vip
        state.max_price = profile.max_price_pln
        state.special_needs = profile.special_needs
        
        # Build info string
        info = [f"Name: {profile.name}"]
        if profile.vip: info.append("Status: VIP")
        if profile.max_price_pln: info.append(f"Max Price: {profile.max_price_pln} PLN")
        if profile.special_needs: info.append(f"Needs: {', '.join(profile.special_needs)}")
        
        return f"✅ CUSTOMER DATA: {' | '.join(info)}"
    
    # === CheckTaxiHistory ===
    if isinstance(action, CheckTaxiHistory):
        # Loop protection
        if state.history_checked:
            problem = state.history_problem or "no issues"
            return f"⚠️ ERROR: History already checked! Result: {problem}. Move to the next step!"
        
        state.history_checked = True
        
        # Construct compound key for DB lookup
        key = (action.customer_id, action.taxi_company)
        record = TAXI_HISTORY.get(key)
        
        if record:
            state.history_problem = record['problem']
            return f"⚠️ HISTORY PROBLEM: {record['problem']}"
        
        return f"✅ History clean with {action.taxi_company}"
        
    return "Next part..."
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.4.4: Dispatcher - część 2

To jest moment, w którym Twój agent przestaje być zabawka, a staje się systemem inżynierskim. Większość tutoriali zakłada "Happy Path" – klient podaje idealny adres, system go przyjmuje. Rzeczywistość tak nie działa.

Gdy klient mówi "na ulicę Prostą", a w mieście są trzy "Proste" (o różnych numerach), zły system wybierze losowo. Dobry system wykryje ryzyko i poprosi o doprecyzowanie.

Twoim zadaniem jest obsłużenie narzędzia `ValidateAddress` w taki sposób, aby agent **zrozumiał**, że natrafił na minę.

### Twoim zadaniem jest:
Rozszerzenie funkcji `dispatch` o logikę walidacji adresu:

1.  **Zabezpieczenie:** Sprawdź, czy adres nie znajduje się już w zbiorze `state.addresses_validated`. Jeśli tak – zablokuj działanie.
2.  **Lookup:** Sprawdź, czy adres w ogóle istnieje w `ADDRESS_DATABASE`.
3.  **Obsługa pułapki (Ambiguity Check):** To jest kluczowe. Sprawdź flagę `ambiguous` w bazie.
    *   **JEŚLI PRAWDA:** Zwróć **ostrzeżenie** (WARNING) i wypisz dostępne opcje z pola `options`.
    *   **WAŻNE:** W tym przypadku **NIE DODAWAJ** adresu do `state.addresses_validated`. Dzięki temu agent będzie wiedział, że zadanie NIE jest wykonane i musi drążyć temat dalej.
4.  **Sukces:** Tylko jeśli adres jest jednoznaczny, dodaj go do zbioru `addresses_validated` i zwróć potwierdzenie.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Logika biznesowa w kodzie

Tu nie chodzi tylko o `if`. Chodzi o sterowanie zachowaniem modelu poprzez stan.

1.  Pobierz info z bazy: `info = ADDRESS_DATABASE.get(action.address)`.
2.  Spójrz do środka: `if info.get("ambiguous"):`.
3.  Jeśli to prawda, Twój return musi "krzyczeć" do modelu: "Hej, nie wiem gdzie jechać! Wybierz jedną z tych opcji: ...".
4.  Model, widząc taki zwrot (i widząc, że w Checkliście adres nadal jest na czerwono ❌), sam wpadnie na to, żeby w następnym kroku zapytać użytkownika/dyspozytora o konkretny numer.

Twój kod powinien wyglądać mniej więcej tak:

```python
    # ...
    if info.get("ambiguous"):
        options = info.get("options", [])
        return f"⚠️ AMBIGUOUS ADDRESS! Options: {', '.join(options)}. You must clarify exact number!"
    
    # Dopiero tutaj zatwierdzamy
    state.addresses_validated.add(action.address)
    return f"✅ Address Validated: {action.address}"
```

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
    # === ValidateAddress ===
    if isinstance(action, ValidateAddress):
        addr = action.address
        
        # Loop protection
        if addr in state.addresses_validated:
            return f"⚠️ ERROR: Address '{addr}' already validated! Move to the next step!"
        
        addr_info = ADDRESS_DATABASE.get(addr)
        if not addr_info:
            return f"❌ Error: Address '{addr}' not found in GIS database"
        
        # AMBIGUITY TRAP HANDLING
        if addr_info.get("ambiguous"):
            options = addr_info.get("options", [])
            # CRITICAL: Do NOT add to validated set! Force the agent to ask clarification.
            return f"⚠️ AMBIGUOUS ADDRESS! Available options: {', '.join(options)}. You must clarify the exact number!"
        
        # Only valid addresses pass here
        state.addresses_validated.add(addr)
        return f"✅ Address Validated: {addr}"
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.4.5*: Dispatcher - rozmowa i słuchanie

Jesteśmy na ostatniej prostej. Mamy narzędzia do czytania danych i walidacji adresów. Teraz musimy obsłużyć samo "mięso" – rozmowę z dyspozytorem.

W systemach agentowych "mówienie" (`Say`) to najłatwiejsza część. Najtrudniejsze jest **słuchanie**. Agent wysyła tekst do symulatora, a w odpowiedzi otrzymuje ludzkie zdanie: *"Taksówka będzie za 15 minut, cena około 40 złotych"*.

**☝️ Ważna uwaga architektoniczna:**
Zwykle w świecie Software 3.0 nie używalibyśmy do tego kodu imperatywnego. Powinniśmy poprosić LLM, aby przeanalizował odpowiedź dyspozytora i zwrócił nam strukturę JSON z polami `eta` i `price`.
Jednak, aby nie komplikować nadmiernie tego ćwiczenia i nie dokładać kolejnych zapytań do API (co wydłużyłoby czas reakcji), zastosujemy tutaj świadome **uproszczenie inżynierskie**.

Użyjemy starego, dobrego Software 1.0 – czyli **wyrażeń regularnych (Regex)** – jako "szybkiego ucha" wewnątrz funkcji `dispatch`. Dzięki temu natychmiast wyłapiemy kluczowe liczby i zaktualizujemy stan, bez konieczności ponownego angażowania modelu.

### Twoim zadaniem jest:
Rozszerzenie funkcji `dispatch` o obsługę akcji `Say`.

1.  **Stan:** Ustaw flagę `state.conversation_started = True`.
2.  **Symulacja:** Wywołaj funkcję `simulate_taxi_dispatcher(action.utterance, goal, state)`. To ona "odegra" rolę człowieka po drugiej stronie.
3.  **Parsowanie (Heurystyka):** Zaimplementuj prostą logikę (zakładamy, że biblioteka `re` jest już zaimportowana), która wyciągnie dane ze zmiennej tekstowej zwróconej przez symulator:
    *   **ETA:** Jeśli znajdziesz liczbę przed słowem "minut", zapisz ją w `state.eta_minutes`.
    *   **Cena:** Jeśli znajdziesz liczbę przed "zł" lub "PLN", zapisz ją w `state.price_quoted`.
    *   **Potwierdzenie:** Jeśli padną słowa potwierdzające (np. "tak", "mamy") w kontekście "fotelika", zaktualizuj `state.special_confirmed`.
    *   **Sukces:** Jeśli padnie hasło "kierowca jedzie" lub "przydzielono", ustaw `state.driver_assigned`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

Mimo że docelowo chcielibyśmy, aby to AI robiło parsing, tutaj zrobimy to "na piechotę" dla szybkości.

```python
    # ... wewnątrz bloku if isinstance(action, Say): ...
    
    response = simulate_taxi_dispatcher(...)
    text = response.lower()

    # Szybkie wyciąganie liczb (Heurystyka)
    # Szukamy wzorca: Cyfry + spacje + "minut"
    time_match = re.search(r'(\d+)\s*minut', text)
    if time_match:
        state.eta_minutes = int(time_match.group(1))
        
    # Analogicznie zrób dla ceny i potwierdzeń...
```

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć ROZWIĄZANIE (Kompletny Dispatcher)
    </summary>
<p>

Oto finalna wersja funkcji `dispatch`, łącząca logikę z zadań 4.4.3, 4.4.4 i obecnego.

```python
def dispatch(action, goal: TaxiBookingGoal, state: ConversationState) -> str:
    
    # === 1. DATA GATHERING (Task 4.4.3) ===
    if isinstance(action, QueryCustomerData):
        if state.customer_fetched:
            return f"⚠️ ERROR: Data already fetched for {state.customer_name}."
        
        profile = CUSTOMERS_DB.get(action.customer_id)
        if not profile: return f"❌ Customer {action.customer_id} not found"
        
        state.customer_fetched = True
        state.customer_name = profile.name
        state.is_vip = profile.vip
        state.max_price = profile.max_price_pln
        state.special_needs = profile.special_needs
        
        info = [f"Name: {profile.name}"]
        if profile.vip: info.append("VIP")
        if profile.special_needs: info.append(f"Needs: {state.special_needs}")
        return f"✅ CUSTOMER DATA: {' | '.join(info)}"

    if isinstance(action, CheckTaxiHistory):
        if state.history_checked:
            return f"⚠️ ERROR: History already checked."
        state.history_checked = True
        key = (action.customer_id, action.taxi_company)
        record = TAXI_HISTORY.get(key)
        if record:
            state.history_problem = record['problem']
            return f"⚠️ HISTORY ISSUE: {record['problem']}"
        return f"✅ HISTORY CLEAN"

    # === 2. VALIDATION (Task 4.4.4) ===
    if isinstance(action, ValidateAddress):
        if action.address in state.addresses_validated:
            return f"⚠️ ERROR: Address '{action.address}' already validated."
        
        info = ADDRESS_DATABASE.get(action.address)
        if not info: return f"❌ Address not found in DB"
        
        if info.get("ambiguous"):
            opts = ", ".join(info.get("options", []))
            return f"⚠️ AMBIGUOUS ADDRESS! Options: {opts}. Ask for clarification!"
        
        state.addresses_validated.add(action.address)
        return f"✅ ADDRESS VALID: {action.address}"

    # === 3. CONVERSATION & PARSING (Task 4.4.5) ===
    if isinstance(action, Say):
        state.conversation_started = True
        
        # Call the simulator
        response = simulate_taxi_dispatcher(action.utterance, goal, state)
        text = response.lower()
        
        # Heuristic Parsing (using re module imported globally)
        
        # Extract ETA
        if "minut" in text:
            match = re.search(r'(\d+)\s*minut', text)
            if match: state.eta_minutes = int(match.group(1))
        
        # Extract Price
        if "zł" in text or "pln" in text:
            match = re.search(r'(\d+)\s*(zł|pln)', text)
            if match: state.price_quoted = int(match.group(1))
            
        # Extract Special Needs Confirmation
        if "fotelik" in text and any(w in text for w in ["tak", "mamy", "jest"]):
            state.special_confirmed = True
            
        # Extract Driver Assignment
        if any(w in text for w in ["jedzie", "przydzielono", "numer boczny"]):
            state.driver_assigned = True
            
        return response

    # === 4. TERMINAL STATES ===
    if isinstance(action, Complete):
        return f"✅ SUCCESS: {action.summary}"
    
    if isinstance(action, Abandon):
        return f"❌ ABORTED: {action.reason}"

    return "❌ Unknown Action"
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.4.6**: Dynamiczny prompting (*Context Engineering*)

To zadanie z dwiema gwiazdkami. Wchodzimy na najwyższy poziom sterowania modelem.

Mamy mózg (`NextStep`) i mamy ręce (`dispatch`). Czego brakuje? **Świadomości sytuacyjnej.**
Model LLM jest bezstanowy. W każdej turze budzi się z amnezją. Twoim zadaniem jest stworzenie funkcji `build_prompt`, która przed każdą decyzją wygeneruje mu **Checklistę (Dashboard)** oraz, co ważne, **podpowie następny logiczny krok**.

Dzięki temu model nie musi zgadywać "co mam teraz zrobić?". On widzi status misji czarno na białym.

### Twoim zadaniem jest:
Napisanie funkcji `build_prompt(goal, state, last_response)`, która zwróci kompletny `system_prompt`. Funkcja musi dynamicznie składać tekst, realizując następującą logikę:

1.  **Status Misji (Checklista):**
    *   Wygeneruj listę z ikonami ✅/❌ dla: danych klienta, historii i walidacji adresów.
    *   Dla negocjacji (ETA, Cena): wyświetl status tylko, jeśli rozmowa się zaczęła.
2.  **Wykrywanie Pułapki (Safety):**
    *   Jeśli w `last_response` pojawi się słowo "NIEJEDNOZNACZNY", wstrzyknij do promptu wielkie ostrzeżenie: *"UWAGA: Adres jest NIEJEDNOZNACZNY! Nie zgaduj, zapytaj!"*.
3.  **Heurystyka (Next Step Hint):**
    *   To jest "sekretny sos". Zamiast liczyć, że model sam wydedukuje kolejność, napisz serię `if/elif`, która na podstawie braków w stanie podpowie mu akcję.
    *   Np. `if not customer_fetched` -> "NASTĘPNY KROK: query_customer...".
4.  **Wstrzykiwanie Danych (Grounding):**
    *   Dodaj sekcję "DANE DO UŻYCIA", gdzie wpiszesz na sztywno `customer_id`, `taxi_company` itp. Poinstruuj model, aby kopiował te wartości, zamiast je wymyślać.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Jak zbudować taki prompt?

Nie pisz jednego wielkiego f-stringa. Zbuduj to modułowo.

1.  Stwórz listę `checklist = []` i dodawaj do niej linie warunkowo.
2.  Stwórz zmienną `next_hint`, która będzie wynikiem prostej maszyny stanów (drabinka `if/elif`).
3.  Na koniec sklej to wszystko w jeden szablon.

Struktura podpowiedzi (`next_hint`):
```python
    next_hint = ""
    if not state.customer_fetched:
        next_hint = "→ NASTĘPNY KROK: query_customer(...)"
    elif not state.history_checked:
        next_hint = "→ NASTĘPNY KROK: check_taxi_history(...)"
    # ... itd
```

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
def build_prompt(goal: TaxiBookingGoal, state: ConversationState, last_response: str = "") -> str:
    
    # Status checklisty
    checklist = []
    
    if state.customer_fetched:
        checklist.append(f"✅ Dane klienta: {state.customer_name} {'(VIP)' if state.is_vip else ''}")
        if state.special_needs:
            checklist.append(f"   → Wymaga: {', '.join(state.special_needs)}")
        if state.max_price:
            checklist.append(f"   → Max cena: {state.max_price} PLN")
    else:
        checklist.append(f"❌ Dane klienta: → query_customer(customer_id='{goal.customer_id}')")
    
    if state.history_checked:
        if state.history_problem:
            checklist.append(f"✅ Historia: ⚠️ PROBLEM: {state.history_problem}")
        else:
            checklist.append(f"✅ Historia: OK (brak problemów)")
    else:
        checklist.append(f"❌ Historia: → check_taxi_history(customer_id='{goal.customer_id}', taxi_company='{goal.taxi_company}')")
    
    # Tracking adresów z obsługą niejednoznacznych
    for addr in [goal.pickup_address, goal.destination]:
        if addr in state.addresses_validated:
            checklist.append(f"✅ Adres: {addr}")
        elif hasattr(state, 'ambiguous_addresses') and addr in getattr(state, 'ambiguous_addresses', {}):
            options = state.ambiguous_addresses[addr]
            checklist.append(f"⚠️ Adres NIEJEDNOZNACZNY: {addr} → ZAPYTAJ dyspozytora który dokładnie!")
        else:
            checklist.append(f"❌ Adres: {addr} → validate_address(address='{addr}')")
    
    if state.conversation_started:
        if state.eta_minutes:
            eta_ok = "✅" if state.eta_minutes <= goal.max_wait_minutes else "❌ ZA DŁUGO!"
            checklist.append(f"{eta_ok} ETA: {state.eta_minutes} min (limit: {goal.max_wait_minutes})")
        
        if state.price_quoted:
            if goal.max_price_pln:
                price_ok = "✅" if state.price_quoted <= goal.max_price_pln else "❌ ZA DROGO!"
                checklist.append(f"{price_ok} Cena: {state.price_quoted} PLN (limit: {goal.max_price_pln})")
        
        if state.special_needs:
            special_ok = "✅" if state.special_confirmed else "❌ NIE POTWIERDZONE!"
            checklist.append(f"{special_ok} Wymagania: {', '.join(state.special_needs)}")
        
        if state.driver_assigned:
            checklist.append(f"✅ Kierowca przydzielony")
    
    checklist_text = "\n".join(checklist)
    
    # Ostatnia odpowiedź
    last_text = ""
    if last_response:
        last_text = f"""
💬 DYSPOZYTOR POWIEDZIAŁ:
"{last_response}"
"""
    
    # Wykryj niejednoznaczny adres w ostatniej odpowiedzi
    ambiguous_warning = ""
    if "NIEJEDNOZNACZNY" in (last_response or ""):
        ambiguous_warning = """
⚠️ UWAGA: Adres jest NIEJEDNOZNACZNY!
→ NIE próbuj walidować opcji osobno!
→ ZAPYTAJ dyspozytora: "Który dokładnie adres - ul. Prosta 5, 15 czy 51?"
→ Użyj akcji: say
"""
    
    # Co robić teraz
    next_hint = ""
    if not state.customer_fetched:
        next_hint = f"→ NASTĘPNY KROK: query_customer(customer_id='{goal.customer_id}')"
    elif not state.history_checked:
        next_hint = f"→ NASTĘPNY KROK: check_taxi_history(customer_id='{goal.customer_id}', taxi_company='{goal.taxi_company}')"
    elif len(state.addresses_validated) < 2:
        missing = [a for a in [goal.pickup_address, goal.destination] if a not in state.addresses_validated]
        if missing:
            next_hint = f"→ NASTĘPNY KROK: validate_address(address='{missing[0]}')"
    elif not state.conversation_started:
        msg_parts = [f"zamówić taksówkę z {goal.pickup_address} do {goal.destination}"]
        if state.special_needs:
            msg_parts.append(f"potrzebuję: {', '.join(state.special_needs)}")
        if state.is_vip and state.history_problem:
            msg_parts.append(f"jestem VIP, ostatnio był problem: {state.history_problem}")
        next_hint = f"→ NASTĘPNY KROK: say(utterance='{'. '.join(msg_parts)}')"
    elif state.driver_assigned and (state.special_confirmed or not state.special_needs):
        next_hint = "→ NASTĘPNY KROK: complete (wszystko OK!)"
    elif state.driver_assigned and state.special_needs and not state.special_confirmed:
        next_hint = f"→ NASTĘPNY KROK: say(utterance='Czy auto ma {state.special_needs[0]}?')"
    elif state.dispatcher_asked_name and not state.name_given:
        if state.special_needs and not state.special_confirmed:
            next_hint = f"→ NASTĘPNY KROK: say(utterance='Zanim podam nazwisko - czy auto ma {state.special_needs[0]}?')"
        else:
            next_hint = f"→ NASTĘPNY KROK: say(utterance='Na nazwisko {state.customer_name}')"

    # Sprawdź czy można kończyć
    can_complete = (
        state.driver_assigned and 
        (not state.special_needs or state.special_confirmed) and
        (not goal.max_price_pln or not state.price_quoted or state.price_quoted <= goal.max_price_pln) and
        (not state.eta_minutes or state.eta_minutes <= goal.max_wait_minutes)
    )
    
    if can_complete:
        next_hint = "→ NASTĘPNY KROK: complete (wszystkie warunki spełnione!)"

    # Dane do użycia w akcjach
    action_data = f"""
📌 DANE DO UŻYCIA W AKCJACH (KOPIUJ DOKŁADNIE!):
   customer_id = "{goal.customer_id}"
   taxi_company = "{goal.taxi_company}"
   pickup = "{goal.pickup_address}"
   destination = "{goal.destination}"
   customer_name = "{state.customer_name or 'nieznane'}"
"""

    return f"""Jesteś agentem zamawiającym taksówkę. Wykonuj akcje krok po kroku.

🎯 CEL: {goal.pickup_address} → {goal.destination}
   Max czas: {goal.max_wait_minutes} min | Max cena: {goal.max_price_pln or 'brak limitu'} PLN | Korporacja: {goal.taxi_company}
{action_data}
📋 STATUS MISJI:
{checklist_text}

{next_hint}
{ambiguous_warning}
{last_text}
══════════════════════════════════════════════════════════════════════════
ZASADY (PRZESTRZEGAJ!):

1. UŻYWAJ DOKŁADNYCH DANYCH z sekcji "DANE DO UŻYCIA" - nie wymyślaj!
   ✗ ŹLE: check_taxi_history(customer_id="CUST_001", taxi_company="VIP")
   ✓ DOBRZE: check_taxi_history(customer_id="{goal.customer_id}", taxi_company="{goal.taxi_company}")

2. NIE POWTARZAJ wykonanych akcji (zobacz ✅ w statusie)

3. Gdy adres NIEJEDNOZNACZNY → ZAPYTAJ dyspozytora, NIE zgaduj!
   ✗ ŹLE: validate_address("ul. Prosta 5") - zgadywanie
   ✓ DOBRZE: say("Który dokładnie adres - ul. Prosta 5, 15 czy 51?")

4. PIERWSZA wiadomość do dyspozytora = WSZYSTKO naraz (trasa + wymagania + VIP jeśli problem)

5. Gdy dyspozytor przydzieli kierowcę I potwierdzi wymagania → complete

6. Gdy warunki NIE spełnione (za drogo/za długo/brak wyposażenia) → abandon

DOSTĘPNE AKCJE:
- query_customer(customer_id: str)
- check_taxi_history(customer_id: str, taxi_company: str)  
- validate_address(address: str)
- say(utterance: str)
- complete(summary: str)
- abandon(reason: str)
══════════════════════════════════════════════════════════════════════════
Co robisz TERAZ? Przeanalizuj status i wykonaj NASTĘPNY KROK."""


```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

## 🧠 Zadanie 4.4.7: Pętla główna

Mamy już wszystkie podzespoły:
1.  **Narzędzia** (Zmysły i ręce)
2.  **NextStep** (Centrum decyzyjny)
3.  **Dispatcher** (Układ nerwowy łączący intencję z działaniem)
4.  **Dynamiczny Prompt** (Świadomość sytuacyjna)

Teraz musimy tchnąć w to życie. Twoim zadaniem jest napisanie funkcji `book_taxi`, która uruchomi pętlę agenta (*Agent Loop*).

To nie jest zwykła pętla `while`. To proces, który w każdej iteracji:
1.  **Obserwuje:** Generuje nowy System Prompt na podstawie aktualnego stanu (`state`).
2.  **Orientuje się:** Wysyła historię do LLM i odbiera obiekt `NextStep`.
3.  **Decyduje:** Sprawdza, czy to koniec misji (`Complete`/`Abandon`).
4.  **Działa:** Przekazuje akcję do `dispatch` i otrzymuje wynik (`feedback`).
5.  **Uczy się:** Dopuje wynik do historii rozmowy.

**Wymaganie inżynierskie:**
Zwróć uwagę na zarządzanie oknem kontekstowym. Rozmowa z dyspozytorem może być długa. Jeśli będziemy tylko doklejać wiadomości, w końcu przekroczymy limit tokenów. Zaimplementuj mechanizm *Sliding Window* – trzymaj *System Prompt* (zawsze aktualny!) na pozycji 0, a potem tylko N ostatnich wiadomości.

### Twoim zadaniem jest:
Złożenie funkcji `book_taxi`, która przyjmuje `customer_id`, `goal` i `scenario`.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć podpowiedź
    </summary>
<p>

### Algorytm pętli:

1.  Zainicjalizuj `state = ConversationState(scenario=scenario)`.
2.  Stwórz listę `messages`.
3.  Rozpocznij pętlę (np. 20 iteracji - bezpiecznik).
4.  **KROK KRYTYCZNY:** W każdej iteracji nadpisz `messages[0]`:
    `messages[0] = {"role": "system", "content": build_prompt(goal, state, last_response)}`
5.  Wywołaj `h.call_api(...)`.
6.  Jeśli akcja to `Complete` lub `Abandon` -> zwróć wynik i przerwij pętlę.
7.  W przeciwnym razie wywołaj `dispatch(...)`.
8.  Dodaj wynik dispatchera do `messages` jako wiadomość `user`.
9.  Przytnij historię (np. zachowaj system prompt + 6 ostatnich wiadomości).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj, aby zobaczyć rozwiązanie
    </summary>
<p>

```python
def book_taxi(
    customer_id: str,
    goal: TaxiBookingGoal,
    scenario: TaxiScenario = TaxiScenario.HAPPY_PATH
) -> Optional[Union[Complete, Abandon]]:
    
    state = ConversationState(scenario=scenario)
    messages = []
    last_response = ""
    if goal.scenario: 
        scenario = goal.scenario
    
    # Inicjalizacja pierwszej wiadomości systemowej
    messages.append({"role": "system", "content": build_prompt(goal, state, last_response)})
    messages.append({"role": "user", "content": "Rozpocznij!"})
    
    for i in range(20):
        print(f"\n{'='*70}")
        print(f"🔄 TURA {i+1}")
        print(f"{'='*70}")
        
        # 1. DYNAMICZNA AKTUALIZACJA PROMPTU
        # To tutaj dzieje się magia - agent "widzi" zmiany w stanie
        messages[0] = {"role": "system", "content": build_prompt(goal, state, last_response)}
        
        # 2. THINKING (LLM Call)
        step = h.call_api(messages=messages, response_model=NextStep)
        
        print(f"\n🧠 Analiza: {step.situation_analysis}")
        print(f"⚡ Akcja: {step.next_action.action}")
        
        # 3. TERMINAL STATES CHECK
        if isinstance(step.next_action, Complete):
            print(f"\n✅ SUKCES: {step.next_action.summary}")
            return step.next_action
        
        if isinstance(step.next_action, Abandon):
            print(f"\n❌ PORAŻKA: {step.next_action.reason}")
            return step.next_action
        
        # 4. ACTING (Dispatch)
        feedback = dispatch(step.next_action, goal, state)
        
        print(f"\n⚙️ WYKONANIE:")
        if isinstance(step.next_action, Say):
            print(f"🤖 Agent: \"{step.next_action.utterance}\"")
            print(f"👤 Dyspozytor: \"{feedback}\"")
            last_response = feedback
        else:
            print(f"📤 {feedback}")
        
        # 5. MEMORY UPDATE
        messages.append({"role": "user", "content": f"Wynik: {feedback}"})
        
        # 6. SLIDING WINDOW (Context Management)
        # Zawsze trzymamy System Prompt [0] + ostatnie 6 wiadomości
        if len(messages) > 8:
            messages = [messages[0]] + messages[-6:]
    
    print("\n⚠️ PRZEKROCZONO LIMIT TUR")
    return None
```
</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE

##  Uruchomienie `Happy Path`

Dobra, mamy to. Mózg (`NextStep`), ręce (`dispatch`), zmysły (narzędzia) i świadomość (`build_prompt`). Wszystko jest na swoim miejscu.

Zanim jednak naciśniesz "Play", jedna **krytyczna uwaga techniczna**:

⚠️ **Funkcja `dispatch`**: Budowaliśmy ją etapami w zadaniach 4.4.3, 4.4.4 i 4.4.5. Upewnij się, że w pamięci notatnika masz załadowaną tę **ostatnią, kompletną wersję** (z zadania 4.4.5), która zawiera w sobie wszystkie gałęzie logiczne: obsługę danych klienta, walidację adresu ORAZ parsowanie rozmowy. Jeśli uruchomisz agenta na niekompletnej funkcji, wywali się na pierwszym zakręcie.

Zakładam, że wszystkie komórki z kodem powyżej są już uruchomione. Jeśli tak – to jest ten moment.

Przekręcamy kluczyk w stacyjce.
Zaczynamy od scenariusza **Happy Path**. Jan Kowalski (VIP), adres "ul. Złota 44" (jest w bazie, jest poprawny), kierunek Lotnisko. Agent powinien gładko przejść przez checklistę i zamówić taryfę.

Sprawdźmy to w praktyce:

In [ ]:
print("\n" + "="*70)
print("TEST 1: HAPPY PATH")
print("="*70)

goal_happy = TaxiBookingGoal(
    customer_id="CUST_001",
    pickup_address="ul. Złota 44",
    destination="Lotnisko Chopina",
    time_preference="now",
    max_wait_minutes=15,
    max_price_pln=150,
    passenger_count=1,
    taxi_company="City Taxi"
)
goal_happy.scenario = TaxiScenario.HAPPY_PATH

result1 = book_taxi(
    customer_id="CUST_001",
    goal=goal_happy
)

## Więcej eksperymentów?

Czy Twój agent potrafi "zrozumieć", że 45 minut czekania to za długo, skoro limit wynosił 15? Czy potrafi z godnością przyjąć odmowę od dyspozytora i zakończyć proces statusem `Abandon`, zamiast pytać w kółko o to samo?

Przygotowałem dla Ciebie dwa dodatkowe testy, które sprawdzą odporność Twojego systemu na stres.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;">
    👉 Kliknij tutaj (1 klik), aby zobaczyć kod testów alternatywnych 👈
    </summary>
<p>

Skopiuj poniższy kod do nowej komórki i uruchom go. Zwróć uwagę na parametr `scenario` w obiekcie `TaxiBookingGoal` – to on steruje zachowaniem naszego symulatora.

```python
print("\n" + "="*70)
print("🚀 TEST 2: NO_CARS_AVAILABLE")
print("="*70)

# Scenariusz: Dyspozytor mówi "przykro mi, brak aut"
no_cars_available = TaxiBookingGoal(
    customer_id="CUST_001",
    pickup_address="ul. Złota 44",
    destination="Lotnisko Chopina",
    time_preference="now",
    max_wait_minutes=15,
    max_price_pln=150,
    passenger_count=1,
    taxi_company="City Taxi",
    scenario=TaxiScenario.NO_CARS_AVAILABLE
)

result2 = book_taxi(customer_id="CUST_001", goal=no_cars_available)


print("\n" + "="*70)
print("🚀 TEST 3: LONG_WAIT_TIME")
print("="*70)

# Scenariusz: Dyspozytor proponuje czas, który przekracza nasz limit (max_wait_minutes)
long_wait_time = TaxiBookingGoal(
    customer_id="CUST_001",
    pickup_address="ul. Złota 44",
    destination="Lotnisko Chopina",
    time_preference="now",
    max_wait_minutes=15, # Limit to 15 min
    max_price_pln=150,
    passenger_count=1,
    taxi_company="City Taxi",
    scenario=TaxiScenario.LONG_WAIT_TIME # Symulator zwróci np. 45 min
)

result3 = book_taxi(customer_id="CUST_001", goal=long_wait_time)
```

</p>
</details>

## Podsumowanie

To była intensywna lekcja, ale masz teraz w rękach działający prototyp inżynierski, a nie tylko zabawkę. Warto na chwilę się zatrzymać i uporządkować to, co właśnie zbudowaliśmy.

Zauważ, że zmieniliśmy całkowicie branżę – z prostej rezerwacji stolika przeskoczyliśmy na logistykę i negocjacje. Mimo to serce systemu pozostało bez zmian. Obiekt decyzyjny nadal pełni rolę centrum dowodzenia, które odpowiada za planowanie i "myślenie" o kolejnym kroku. To dowód na uniwersalność tej architektury.

Z drugiej strony, ciężar pracy wykonawczej przeniósł się mocno do warstwy kodu. Funkcja dispatch przestała być prostym przełącznikiem, a stała się strażnikiem stabilności. To brutalna prawda o systemach agentowych: model jest świetny w planowaniu, ale to kod musi trzymać w ryzach walidację danych i obsługę błędów. Bez tego twardego fundamentu system by się posypał przy pierwszej literówce w adresie.

Wprowadziliśmy też pojęcie stanu rozmowy. Te dodatkowe struktury pamięci będą się naturalnie rozrastać w miarę komplikowania projektów, ale zasada pozostanie ta sama: model jest bezstanowy, więc my musimy dostarczać mu kontekst.

W kolejnej lekcji zmienimy otoczenie na zupełnie nowy przykład. Czekam na Ciebie z kolejnym wyzwaniem, aby ostatecznie utrwalić ten wzorzec w praktyce, zanim pójdziemy dalej.